# 데이터 EDA & 전처리

---

In [11]:
# 데이터 호출

data <- read.csv(
  "data.csv",
  stringsAsFactors = FALSE,
  na.strings = c("", "NA", "N/A")
)

In [12]:
dim(data)
head(data)

[1] 71351    32

,spkid,full_name,name,neo,pha,sats,H,G,PC,diameter,⋯,n,tp,tp_cal,per,per_y,moid,moid_ld,class,moid_jup,equinox
,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<dbl>,<dbl>,<lgl>,<dbl>,⋯,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>
1,20000132,132 Aethra (A873 LA),Aethra,N,N,0,8.96,NA,NA,42.870,⋯,0.2335,2460516,2024-07-24.7,1540,4.22,0.782,304.0,MCA,2.21,J2000
2,20000391,391 Ingeborg (A894 VB),Ingeborg,N,N,0,10.85,NA,NA,15.751,⋯,0.2788,2460826,2025-05-30.0,1290,3.54,0.645,251.0,MCA,2.54,J2000
3,20000433,433 Eros (A898 PA),Eros,Y,N,0,10.39,0.46,NA,16.840,⋯,0.5598,2461089,2026-02-17.3,643,1.76,0.148,57.7,AMO,3.29,J2000
4,20000475,475 Ocllo (A901 PA),Ocllo,N,N,0,11.54,NA,NA,17.830,⋯,0.2360,2461442,2027-02-05.8,1530,4.18,0.674,262.0,MCA,2.10,J2000
5,20000512,512 Taurinensis (A903 MC),Taurinensis,N,N,0,10.76,NA,NA,23.090,⋯,0.3043,2461448,2027-02-11.6,1180,3.24,0.651,253.0,MCA,2.72,J2000
6,20000699,699 Hela (A910 LC),Hela,N,N,0,11.45,NA,NA,NA,⋯,0.2331,2460668,2024-12-23.3,1540,4.23,0.628,244.0,MCA,2.10,J2000


In [13]:
data <- read.csv(
  "data.csv",
  stringsAsFactors = FALSE,
  na.strings = c("", "NA", "N/A")
)

# feature별 기본 요약
feature_summary <- data.frame(
  feature = names(data),
  type = sapply(data, function(x) class(x)[1]),
  n = nrow(data),
  missing_n = sapply(data, function(x) sum(is.na(x))),
  missing_pct = round(sapply(data, function(x) mean(is.na(x)) * 100), 2),
  unique_n = sapply(data, function(x) length(unique(x[!is.na(x)]))),
  stringsAsFactors = FALSE
)

feature_summary$constant <- feature_summary$unique_n <= 1

feature_summary

,feature,type,n,missing_n,missing_pct,unique_n,constant
,<chr>,<chr>,<int>,<int>,<dbl>,<int>,<lgl>
spkid,spkid,integer,71351,0,0.00,71351,FALSE
full_name,full_name,character,71351,0,0.00,71351,FALSE
name,name,character,71351,70882,99.34,469,FALSE
neo,neo,character,71351,0,0.00,2,FALSE
pha,pha,character,71351,773,1.08,2,FALSE
sats,sats,integer,71351,0,0.00,3,FALSE
H,H,numeric,71351,58,0.08,1859,FALSE
G,G,numeric,71351,71337,99.98,12,FALSE
PC,PC,logical,71351,71351,100.00,0,TRUE


In [14]:
# 제거할 feature 지정
drop_cols <- c(
  "spkid", "full_name", "name",       # 식별자: 천체 이름/번호라 예측 feature로 부적절

  "G", "PC", "diameter", "extent",
  "albedo", "rot_per", "BV",          # 결측률 90% 이상: na.omit 전에 열 단위 제거

  "orbit_id",                         # 내부 궤도해 식별자: 물리/궤도 측정값이 아님

  "equinox",                          # 모든 값이 동일한 상수 feature

  "tp",                               # 근일점 통과 시각: 현재 목적의 핵심 궤도 형상 feature로 보기 어려움

  "tp_cal",                           # tp의 달력형 변환값

  "moid",                             # 궤도 요소로부터 계산되는 값이며 PHA 정의에 직접 사용됨

  "moid_ld",                          # moid의 단위 변환값

  "class",                            # 궤도 요소를 바탕으로 붙은 분류 라벨

  "q", "ad",                          # a, e로부터 유도 가능: q = a * (1 - e), ad = a * (1 + e)

  "n", "per", "per_y",                # 공전 속도/공전 주기 계열 중복: n = 360 / per, per_y는 per 단위 변환

  "H"                                 # PHA 정의에 직접 들어가므로 leakage 방지 목적에서 제거
)

# 실제 데이터에 존재하는 열만 제거 대상으로 사용
drop_cols <- drop_cols[drop_cols %in% names(data)]

# 제거된 feature 출력
cat("제거할 feature:\n")
print(drop_cols)

# feature 제거
data_clean <- data[, !(names(data) %in% drop_cols)]

# 남은 feature 출력
cat("\n남은 feature:\n")
print(names(data_clean))

# 남은 데이터에서 결측치가 있는 행 제거
data_clean <- na.omit(data_clean)

# 최종 데이터 크기 확인
dim(data_clean)

제거할 feature:
 [1] "spkid"     "full_name" "name"      "G"         "PC"        "diameter" 
 [7] "extent"    "albedo"    "rot_per"   "BV"        "orbit_id"  "equinox"  
[13] "tp"        "tp_cal"    "moid"      "moid_ld"   "class"     "q"        
[19] "ad"        "n"         "per"       "per_y"     "H"        

남은 feature:
[1] "neo"      "pha"      "sats"     "e"        "a"        "i"        "om"      
[8] "w"        "moid_jup"


[1] 70578     9

In [15]:
write.csv(
  data_clean,
  "data_clean.csv",
  row.names = FALSE
)

---

In [16]:
# 각도형 feature 변환 및 학습 데이터 구성

library(randomForest)
library(keras3)

set.seed(2026)

In [18]:
# Y = 1, N = 0
data_model <- data_clean

data_model$neo_bin <- ifelse(data_model$neo == "Y", 1, 0)
data_model$pha_bin <- ifelse(data_model$pha == "Y", 1, 0)

In [19]:
# om, w는 각도형 변수이므로 sin/cos 변환

data_model$om_rad <- data_model$om * pi / 180
data_model$w_rad  <- data_model$w  * pi / 180

data_model$om_sin <- sin(data_model$om_rad)
data_model$om_cos <- cos(data_model$om_rad)

data_model$w_sin <- sin(data_model$w_rad)
data_model$w_cos <- cos(data_model$w_rad)

In [20]:
# 최종 입력 feature

feature_cols <- c(
  "sats",
  "e",
  "a",
  "i",
  "moid_jup",
  "om_sin",
  "om_cos",
  "w_sin",
  "w_cos"
)

data_model <- data_model[, c(feature_cols, "neo_bin", "pha_bin")]

In [21]:
# train / test split
# neo 기준으로 stratified split

stratified_split <- function(y, train_ratio = 0.8, seed = 2026) {
  set.seed(seed)
  
  idx_0 <- which(y == 0)
  idx_1 <- which(y == 1)
  
  train_0 <- sample(idx_0, size = floor(length(idx_0) * train_ratio))
  train_1 <- sample(idx_1, size = floor(length(idx_1) * train_ratio))
  
  sort(c(train_0, train_1))
}

train_idx <- stratified_split(data_model$neo_bin, train_ratio = 0.8)

train_data <- data_model[train_idx, ]
test_data  <- data_model[-train_idx, ]

In [22]:
# 딥러닝 모델 학습을 위해 numeric feature를 표준화

scale_params <- data.frame(
  feature = feature_cols,
  mean = sapply(train_data[, feature_cols], mean),
  sd = sapply(train_data[, feature_cols], sd),
  row.names = NULL
)

scale_params$sd[scale_params$sd == 0] <- 1

scale_features <- function(df, scale_params) {
  for (col in scale_params$feature) {
    mu <- scale_params$mean[scale_params$feature == col]
    sig <- scale_params$sd[scale_params$feature == col]
    df[[col]] <- (df[[col]] - mu) / sig
  }
  df
}

train_data <- scale_features(train_data, scale_params)
test_data  <- scale_features(test_data, scale_params)


# 확인
cat("Train size:", nrow(train_data), "\n")
cat("Test size :", nrow(test_data), "\n")

cat("\nTrain neo distribution:\n")
print(prop.table(table(train_data$neo_bin)))

cat("\nTrain pha distribution:\n")
print(prop.table(table(train_data$pha_bin)))

Train size: 56462 
Test size : 14116 

Train neo distribution:

        0         1 
0.4099571 0.5900429 

Train pha distribution:

         0          1 
0.96379866 0.03620134 


---

In [23]:
# 평가 함수

evaluate_binary <- function(actual, prob, threshold = 0.5) {
  pred <- ifelse(prob >= threshold, 1, 0)
  
  actual <- factor(actual, levels = c(0, 1))
  pred <- factor(pred, levels = c(0, 1))
  
  cm <- table(Actual = actual, Predicted = pred)
  
  tn <- cm["0", "0"]
  fp <- cm["0", "1"]
  fn <- cm["1", "0"]
  tp <- cm["1", "1"]
  
  accuracy <- (tp + tn) / sum(cm)
  precision <- ifelse((tp + fp) == 0, NA, tp / (tp + fp))
  recall <- ifelse((tp + fn) == 0, NA, tp / (tp + fn))
  specificity <- ifelse((tn + fp) == 0, NA, tn / (tn + fp))
  f1 <- ifelse(
    is.na(precision) | is.na(recall) | (precision + recall) == 0,
    NA,
    2 * precision * recall / (precision + recall)
  )
  
  result <- data.frame(
    accuracy = round(accuracy, 4),
    precision = round(precision, 4),
    recall = round(recall, 4),
    specificity = round(specificity, 4),
    f1 = round(f1, 4)
  )
  
  list(
    confusion_matrix = cm,
    metrics = result
  )
}


# 클래스 불균형 대응용 class weight 계산
make_class_weight <- function(y) {
  tab <- table(factor(y, levels = c(0, 1)))
  n <- sum(tab)
  
  list(
    "0" = as.numeric(n / (2 * tab["0"])),
    "1" = as.numeric(n / (2 * tab["1"]))
  )
}

In [26]:
# Random Forest 모델

# neo 예측

rf_neo <- randomForest(
  x = train_data[, feature_cols],
  y = factor(train_data$neo_bin, levels = c(0, 1), labels = c("N", "Y")),
  ntree = 500,
  importance = TRUE
)

neo_rf_prob <- predict(
  rf_neo,
  test_data[, feature_cols],
  type = "prob"
)[, "Y"]

neo_rf_pred <- ifelse(neo_rf_prob >= 0.5, 1, 0)

rf_neo_eval <- evaluate_binary(
  actual = test_data$neo_bin,
  prob = neo_rf_prob,
  threshold = 0.5
)

cat("Random Forest - NEO evaluation\n")
print(rf_neo_eval$confusion_matrix)
print(rf_neo_eval$metrics)


# ------------------------------------------------------------
# 5.2 2단계 모델: 실제 neo = Y인 train 데이터만 사용하여 pha 예측
# ------------------------------------------------------------

train_neo_y <- train_data[train_data$neo_bin == 1, ]

rf_pha <- randomForest(
  x = train_neo_y[, feature_cols],
  y = factor(train_neo_y$pha_bin, levels = c(0, 1), labels = c("N", "Y")),
  ntree = 500,
  importance = TRUE
)

Random Forest - NEO evaluation
      Predicted
Actual    0    1
     0 5760   27
     1   19 8310
  accuracy precision recall specificity     f1
1   0.9967    0.9968 0.9977      0.9953 0.9972


In [27]:
# neo 예측값이 N이면 pha는 N으로 처리
# neo 예측값이 Y인 경우에만 pha 모델 적용

pha_rf_prob_final <- rep(0, nrow(test_data))
neo_y_pred_idx_rf <- which(neo_rf_pred == 1)

if (length(neo_y_pred_idx_rf) > 0) {
  pha_rf_prob_final[neo_y_pred_idx_rf] <- predict(
    rf_pha,
    test_data[neo_y_pred_idx_rf, feature_cols],
    type = "prob"
  )[, "Y"]
}

pha_rf_eval_final <- evaluate_binary(
  actual = test_data$pha_bin,
  prob = pha_rf_prob_final,
  threshold = 0.5
)

cat("Random Forest - Final PHA pipeline evaluation\n")
print(pha_rf_eval_final$confusion_matrix)
print(pha_rf_eval_final$metrics)

Random Forest - Final PHA pipeline evaluation
      Predicted
Actual     0     1
     0 13605     9
     1   487    15
  accuracy precision recall specificity    f1
1   0.9649     0.625 0.0299      0.9993 0.057


In [28]:
# 실제 neo = Y인 test 데이터 내부에서 pha 모델만 평가

test_neo_y <- test_data[test_data$neo_bin == 1, ]

pha_rf_prob_stage2 <- predict(
  rf_pha,
  test_neo_y[, feature_cols],
  type = "prob"
)[, "Y"]

pha_rf_eval_stage2 <- evaluate_binary(
  actual = test_neo_y$pha_bin,
  prob = pha_rf_prob_stage2,
  threshold = 0.5
)

cat("Random Forest - PHA evaluation within actual NEO objects\n")
print(pha_rf_eval_stage2$confusion_matrix)
print(pha_rf_eval_stage2$metrics)


# ------------------------------------------------------------
# 5.5 변수 중요도
# ------------------------------------------------------------

cat("Random Forest - NEO variable importance\n")
print(importance(rf_neo))

cat("Random Forest - PHA variable importance\n")
print(importance(rf_pha))

Random Forest - PHA evaluation within actual NEO objects
      Predicted
Actual    0    1
     0 7818    9
     1  487   15
  accuracy precision recall specificity    f1
1   0.9404     0.625 0.0299      0.9989 0.057
Random Forest - NEO variable importance
                  N          Y MeanDecreaseAccuracy MeanDecreaseGini
sats       3.941629  -1.259781             3.105428         1.215867
e        473.075101 447.785563           551.282428     10901.722484
a        275.827714 112.365790           195.297293     10764.044390
i         35.633510  23.936194            37.881404       979.367844
moid_jup  34.950328  28.582185            40.105684      4069.216324
om_sin    15.964353  15.135626            21.368866       122.626358
om_cos    17.215803  16.868677            23.444500       134.328407
w_sin     18.403168  19.442639            25.647000       181.412308
w_cos     19.523082  15.860278            23.899448       162.540894
Random Forest - PHA variable importance
              

In [30]:
# Keras/TensorFlow 기반 간단한 딥러닝 모델

# matrix 변환
x_train_neo <- as.matrix(train_data[, feature_cols])
y_train_neo <- train_data$neo_bin

x_test <- as.matrix(test_data[, feature_cols])

# MLP 모델 생성 함수
build_mlp <- function(input_dim) {
  model <- keras_model_sequential(input_shape = input_dim) |>
    layer_dense(units = 32, activation = "relu") |>
    layer_dropout(rate = 0.25) |>
    layer_dense(units = 16, activation = "relu") |>
    layer_dropout(rate = 0.25) |>
    layer_dense(units = 1, activation = "sigmoid")
  
  model |>
    compile(
      optimizer = optimizer_adam(learning_rate = 0.001),
      loss = "binary_crossentropy",
      metrics = c("accuracy")
    )
  
  model
}

In [31]:
# DL 모델: neo 예측

class_weight_neo <- make_class_weight(y_train_neo)

dl_neo <- build_mlp(input_dim = length(feature_cols))

history_neo <- dl_neo |>
  fit(
    x = x_train_neo,
    y = y_train_neo,
    epochs = 80,
    batch_size = 256,
    validation_split = 0.2,
    class_weight = class_weight_neo,
    verbose = 1,
    callbacks = list(
      callback_early_stopping(
        monitor = "val_loss",
        patience = 10,
        restore_best_weights = TRUE
      )
    )
  )

neo_dl_prob <- as.numeric(
  predict(dl_neo, x_test, verbose = 0)
)

neo_dl_pred <- ifelse(neo_dl_prob >= 0.5, 1, 0)

dl_neo_eval <- evaluate_binary(
  actual = test_data$neo_bin,
  prob = neo_dl_prob,
  threshold = 0.5
)

cat("Deep Learning - NEO evaluation\n")
print(dl_neo_eval$confusion_matrix)
print(dl_neo_eval$metrics)

Deep Learning - NEO evaluation
      Predicted
Actual    0    1
     0 5754   33
     1   14 8315
  accuracy precision recall specificity     f1
1   0.9967     0.996 0.9983      0.9943 0.9972


In [32]:
# 실제 neo = Y인 train 데이터만 사용하여 pha 예측

x_train_pha <- as.matrix(train_neo_y[, feature_cols])
y_train_pha <- train_neo_y$pha_bin

class_weight_pha <- make_class_weight(y_train_pha)

dl_pha <- build_mlp(input_dim = length(feature_cols))

history_pha <- dl_pha |>
  fit(
    x = x_train_pha,
    y = y_train_pha,
    epochs = 100,
    batch_size = 256,
    validation_split = 0.2,
    class_weight = class_weight_pha,
    verbose = 1,
    callbacks = list(
      callback_early_stopping(
        monitor = "val_loss",
        patience = 12,
        restore_best_weights = TRUE
      )
    )
  )

In [33]:
# 2단계 DL 추론
# neo 예측값이 N이면 pha는 N으로 처리
# neo 예측값이 Y인 경우에만 pha 모델 적용

pha_dl_prob_final <- rep(0, nrow(test_data))
neo_y_pred_idx_dl <- which(neo_dl_pred == 1)

if (length(neo_y_pred_idx_dl) > 0) {
  pha_dl_prob_final[neo_y_pred_idx_dl] <- as.numeric(
    predict(
      dl_pha,
      as.matrix(test_data[neo_y_pred_idx_dl, feature_cols]),
      verbose = 0
    )
  )
}

pha_dl_eval_final <- evaluate_binary(
  actual = test_data$pha_bin,
  prob = pha_dl_prob_final,
  threshold = 0.5
)

cat("Deep Learning - Final PHA pipeline evaluation\n")
print(pha_dl_eval_final$confusion_matrix)
print(pha_dl_eval_final$metrics)

Deep Learning - Final PHA pipeline evaluation
      Predicted
Actual     0     1
     0 10418  3196
     1   109   393
  accuracy precision recall specificity     f1
1   0.7659    0.1095 0.7829      0.7652 0.1921


In [34]:
# 실제 neo = Y인 test 데이터 내부에서 pha 모델만 평가

pha_dl_prob_stage2 <- as.numeric(
  predict(
    dl_pha,
    as.matrix(test_neo_y[, feature_cols]),
    verbose = 0
  )
)

pha_dl_eval_stage2 <- evaluate_binary(
  actual = test_neo_y$pha_bin,
  prob = pha_dl_prob_stage2,
  threshold = 0.5
)

cat("Deep Learning - PHA evaluation within actual NEO objects\n")
print(pha_dl_eval_stage2$confusion_matrix)
print(pha_dl_eval_stage2$metrics)

Deep Learning - PHA evaluation within actual NEO objects
      Predicted
Actual    0    1
     0 4633 3194
     1  108  394
  accuracy precision recall specificity     f1
1   0.6036    0.1098 0.7849      0.5919 0.1927


In [35]:
# 결과

result_compare <- data.frame(
  neo_actual = ifelse(test_data$neo_bin == 1, "Y", "N"),
  pha_actual = ifelse(test_data$pha_bin == 1, "Y", "N"),
  
  neo_rf_prob = neo_rf_prob,
  neo_rf_pred = ifelse(neo_rf_pred == 1, "Y", "N"),
  pha_rf_prob = pha_rf_prob_final,
  pha_rf_pred = ifelse(pha_rf_prob_final >= 0.5, "Y", "N"),
  
  neo_dl_prob = neo_dl_prob,
  neo_dl_pred = ifelse(neo_dl_pred == 1, "Y", "N"),
  pha_dl_prob = pha_dl_prob_final,
  pha_dl_pred = ifelse(pha_dl_prob_final >= 0.5, "Y", "N")
)

head(result_compare)


# 모델별 핵심 지표 요약
model_summary <- rbind(
  data.frame(
    model = "RF",
    target = "neo",
    rf_neo_eval$metrics
  ),
  data.frame(
    model = "RF",
    target = "pha_final_pipeline",
    pha_rf_eval_final$metrics
  ),
  data.frame(
    model = "RF",
    target = "pha_within_actual_neo",
    pha_rf_eval_stage2$metrics
  ),
  data.frame(
    model = "DL",
    target = "neo",
    dl_neo_eval$metrics
  ),
  data.frame(
    model = "DL",
    target = "pha_final_pipeline",
    pha_dl_eval_final$metrics
  ),
  data.frame(
    model = "DL",
    target = "pha_within_actual_neo",
    pha_dl_eval_stage2$metrics
  )
)

model_summary

,neo_actual,pha_actual,neo_rf_prob,neo_rf_pred,pha_rf_prob,pha_rf_pred,neo_dl_prob,neo_dl_pred,pha_dl_prob,pha_dl_pred
,<chr>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>,<dbl>,<chr>
1,N,N,0.000,N,0,N,0.0015537181,N,0,N
2,N,N,0.002,N,0,N,0.0010270567,N,0,N
6,N,N,0.004,N,0,N,0.0012169273,N,0,N
13,N,N,0.000,N,0,N,0.0002610534,N,0,N
18,N,N,0.000,N,0,N,0.0007162697,N,0,N
23,N,N,0.000,N,0,N,0.0013734420,N,0,N


model,target,accuracy,precision,recall,specificity,f1
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
RF,neo,0.9967,0.9968,0.9977,0.9953,0.9972
RF,pha_final_pipeline,0.9649,0.6250,0.0299,0.9993,0.0570
RF,pha_within_actual_neo,0.9404,0.6250,0.0299,0.9989,0.0570
DL,neo,0.9967,0.9960,0.9983,0.9943,0.9972
DL,pha_final_pipeline,0.7659,0.1095,0.7829,0.7652,0.1921
DL,pha_within_actual_neo,0.6036,0.1098,0.7849,0.5919,0.1927


In [36]:
write.csv(
  result_compare,
  "prediction_result_compare.csv",
  row.names = FALSE
)

write.csv(
  model_summary,
  "model_summary.csv",
  row.names = FALSE
)